<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/sts_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta OAuth-STS Acces Demo

This notebook demonstrates the complete **STS** flow for accessing SAAS resources using the Okta AI SDK.


## The 3-Step OAuth STS Flow

1. **Get ID Token** - Authenticate as a user to obtain an ID Token
1. **Exchange for a Resource Token** - Exchange the user's token for stored resource access token
1. **Resource API Access** - Use the resource's access token to call the resource API.
---

### Prerequisites

Before running this notebook, you need:

1. **Okta Organization** with:
   - Resource Application server configured

2. **Agent Principal** with:
   - Principal ID (agent identifier)
   - Private JWK (RSA key pair for JWT bearer assertion)

3. **User**:
   - Valid Okta User to authenticate and obtain an ID token.

4. **Execution order for extended steps in this notebook**:
   - Run setup and Steps 1 before running Steps 2-3.
   - Step 2 expects `ID_TOKEN`, `okta_sdk`, and (optionally) `REFRESH_TOKEN` to already exist.

## Setup and Installation


> #### <br>**IMPORTANT**
> 
> **Training Notebook Note**
> - This notebook intentionally includes some sample-code shortcuts for readability and step-by-step learning.
> - For production: store secrets outside notebooks, enforce full token verification, and package helper logic into tested modules.
> - Before publishing to GitHub, remove local-only utilities and any environment-specific scaffolding.<br><br>

### 1. Install the Okta Python SDK

In [ ]:
# Install the Okta AI SDK from PyPI

%pip install --upgrade okta-client-python

### 2. Set Configuration

Set _all_ your configuration variables and _run the cell._

In [ ]:
# @title { display-mode: "form", vertical-output: true }
# @markdown _Enter your configuration variables here and click the 'run' to validate._
# @markdown <br><br> These will be used throughout the notebook.

# OKTA_DOMAIN = 'https://{your_domain}.oktapreview.com' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
# @markdown <br>
OKTA_DOMAIN = '' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
# @markdown <br>

# @markdown ---
# @markdown ##### Principal/Agent Configuration
# @markdown ---
PRINCIPAL_ID = '' # @param {"type":"string","placeholder":"Enter your agent identifier"}
PRINCIPAL_SCOPES = [] # @param {"type":"raw","placeholder": "Enter the scopes your agent should request."}

# @markdown <sup><em>If you are using client secret authentication, enter your agent secret below.</em></sup>
PRINCIPAL_SECRET = '' # @param {"type":"string","placeholder":"Enter your agent's client secret"}
# @markdown <sup><u>or...</u> <em>If you are using private key authentication, enter your agent's private JWK below.</em></sup>
PRINCIPAL_PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Client (App) configuration
# @markdown ---
REDIRECT_URI = 'http://localhost:8080/authorization-code/callback' # @param {"type":"string","placeholder":"Enter application redirect URI"}
# @markdown <br>

# @markdown ---
# @markdown ##### Resource Configuration
# @markdown ---
RESOURCE_INDICATOR = '' # @param {"type":"string","placeholder":"Enter the resource indicator"}
# @markdown <br>

# @markdown ---
SDK_DEBUG_ENABLED = False # @param {"type":"boolean"}

# ===== Uncomment only for local dev and comment out the next section =====
# from pathlib import Path
# import sys

# cwd = Path.cwd()
# repo_root = cwd if (cwd / "utils.py").exists() else cwd.parent
# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# import utils
# validate_config = utils.validate_config
# globals().setdefault("Debugger", utils.Debugger)
# ==============

# Import utility functions for validation and other operations
# === Uncomment this section to load utils.py from GitHub if you are not running this notebook locally ===
import requests
from typing import Callable
url = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/utils.py"

response = requests.get(url)
if response.status_code == 200:
    # This executes the code inside utils.py
    exec(response.text)
    print("Successfully loaded utils.py from GitHub!")
else:
    raise Exception(f"Failed to load utils.py. Please contact a code owner -- this is not good!")

validate_config: Callable[[dict, str], None] | None = None
# ========

# Perform validation of the configuration variables
# Function will always exist in utils.py, but we check for its existence to avoid errors if the file fails to load.
if validate_config:
    try:
        validate_config(locals(), "sts");
    except ValueError as e:
        print(e);
        raise SystemExit("❌ Configuration validation failed. Please fix the above errors and re-run the cell.");

print("\n✅ All configuration variables validated successfully!");
print(f"    Okta Domain: {OKTA_DOMAIN}");

print(f"    Principal ID: {PRINCIPAL_ID}");
print(f"    Principal Scopes: {PRINCIPAL_SCOPES}");

print(f"    Resource Indicator: {RESOURCE_INDICATOR}");

global ISSUER
ISSUER = f"{OKTA_DOMAIN}/oauth2"

### 3. Initialize the Okta Python SDK.

In [ ]:
# Initialize Okta SDK
from okta_client.authfoundation import OAuth2Client, OAuth2ClientConfiguration, ClientSecretAuthorization, LocalKeyProvider
from okta_client.authfoundation.oauth2.jwt_bearer_claims import JWTBearerClaims
from okta_client.authfoundation.oauth2.client_authorization import ClientAssertionAuthorization
from okta_client.oauth2auth import AuthorizationCodeContext, AuthorizationCodeFlow, Prompt

STATE = globals().setdefault("STATE", {});

def build_client_authorization(client_id: str, client_secret: str, private_jwk: dict, audience: str, label: str):
    """Build client auth using private key if available, otherwise client secret."""
    if private_jwk and isinstance(private_jwk, dict) and private_jwk.get("kid"):
        print(f"\n✅ Using private key auth for {label} (kid={private_jwk.get('kid')})");
        return ClientAssertionAuthorization(
            assertion_claims=JWTBearerClaims(
                issuer=client_id,
                subject=client_id,
                audience=audience,
                expires_in=300
            ),
            key_provider=LocalKeyProvider(
                key=private_jwk,
                algorithm=private_jwk.get("alg", "RS256"),
                key_id=private_jwk.get("kid")
            )
        );

    print(f"\n✅ Using client secret auth for {label}")
    return ClientSecretAuthorization(
        id=client_id,
        secret=client_secret
    );

client_authz = build_client_authorization(
    client_id=PRINCIPAL_ID,
    client_secret=PRINCIPAL_SECRET,
    private_jwk=PRINCIPAL_PRIVATE_JWK,
    audience=f"{OKTA_DOMAIN}/oauth2/v1/token",
    label="client-principal"
);

# Initialize the SDK with the appropriate configuration, including the issuer for the custom authorization server.
sdk_config = OAuth2ClientConfiguration(
    issuer=ISSUER,
    scope=["openid", "profile"],
    redirect_uri=REDIRECT_URI,
    client_authorization=client_authz
);

okta_sdk = OAuth2Client(configuration=sdk_config);
STATE["okta_sdk"] = okta_sdk;

print("✅ Okta SDK initialized!")

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("⚠️ Debugger not loaded; continuing without SDK listener.");
    else:
        print("ℹ️ SDK Debugging enabled; adding debugger listener to Okta SDK.");
        okta_sdk.listeners.add(debugger_cls()); # type: ignore

---

## STEP 1: Obtain ID Token

If you don't have an ID token yet, use this section to obtain one through OAuth 2.0 authorization code flow.

### Important:
This step uses the **Org Authorization Server** (not a custom authorization server). The ID token will be issued directly by your Okta domain:
- Issuer: `https://your-domain.okta.com`
- Endpoint: `/oauth2/v1/authorize` and `/oauth2/v1/token`

This is the correct approach for obtaining the initial user ID token that will be used in the ID-JAG flow.

### The Flow:

1. **Build authorization URL:** User authenticates in browser (Org Authorization Server)
1. **Copy redirect URL from browser:** Copy the entire redirect URL after authentication.
1. **Exchange code for tokens:** Get ID token with issuer = Okta domain

[![](https://mermaid.ink/img/pako:eNptkk-PmzAQxb_KaHrZVdmIP3FgOawU0es2hzQ9rLgY8BIrwaaD2baJ8t07xt2qUgtC8vj9_Dx--Iqt7RSWOKlvszKt-qRlT3KoDfAjZ2fNPDSKQj1KcrrVozQODiAnOEz_kyovVWeteLwdx3-B3Xbvkd3JSdhRD9vZHS-wV_Tm7fh1pFoH1Dd3cQRJWkSQCnHvhdp9tk4B6f7owL7CoYS9UyMkQazd4eHpqSoXS95ft5LpNy2hIft96TZgFWPcRgAt6Yt02hqoOA34CF_sSRkgn8nk3pcw_hDM_bFh27ZqmgLqEWU6jLAn3WHpaFYRDooG6Uu8-gxq5JYGVWPJw07SqcYozPvimXf20rL0L_yrJC2bs5q8eA1Z1tjI9tSTnU0X7D5kWVajV2-1uXEfnPOLtcN7K4z2Ryxf5Xniah47juX3r_4zS3wCRRWbOiwTIcTiguUVf2CZinQlRLrO8jzhL2fxJ1OJWK3zOF9vNkn2WDyKW4SXZdt4VeTJOk6LJM3yON4UeYSq087Sc7hwy727_QKeAc6P?type=png)](https://mermaid.live/edit#pako:eNptks1u2zAQhF9lsb20iGKY-rcOAQz1mvrguodAF0piZMKWqK6oNLHhd-9SbIICrXjhcj4OlyNesTGtwgIn9XNWQ6O-atmR7KsB-JOzNcPc14p8PUqyutGjHCwcQE5wmP4nlU4qz1rxfDuO_wK77d4hu5OVsKMOtrM9XmCv6MXZ8bCkGgvU1Z_XAYgwDyBMki9OqOw3YxWQ7o4WzDMcCthbNYLwYmUP9w8PZbFY8vm6kUy_aAk1mV9Ltx4rGeM2PGhIX6TVZoCS04A7-G5OagBymUz2fQvj997cXRu2TaOmyaMOUUOLAXakWywszSrAXlEvXYlXl0GF3FKvKix42ko6VRj4dVc88slOWrb-hf-QpGV9VpMTrz7LCmvZnDoy89B6u09RFFXo1Fs13LgPzvnJmP69FUa7IxbP8jxxNY8tx_LnV3-sEt9AUcmmFguRxNHigsUVX7mOwlUW50m2ESLfhGHK6hsWyWYlsizO1iKO83iTJbcAL8u5YiXiVGTrLE_DPM3TKA9QtdoaevQvbnl4t99r9c7a)

### ① Build Authorization URL and Authenticate via the browser

Run the following cell and then click the generated button to open the authorization URL in a new browser tab.

In [ ]:
from IPython.display import HTML, display # Ensure display is imported for the HTML button

# ID Token (will be obtained in STEP 0, or provide your own)
ID_TOKEN = None  # Leave as None to obtain via authorization code flow
ACCESS_TOKEN = None
REFRESH_TOKEN = None

async def authorize():

  try:
    # Build the authorization URL
    print("\n⏳ Building authorization URL...");
    authorization_context = AuthorizationCodeContext(
        prompt=Prompt.LOGIN,
    );

    if "okta_sdk" not in globals() and not STATE.get("okta_sdk"):
        raise SystemExit("❌ Missing Okta SDK. Run 'Setup' step 3 (Initialize the Okta SDK) first.");

    global auth_flow;
    auth_flow = AuthorizationCodeFlow(client=okta_sdk);
    STATE["auth_flow"] = auth_flow;

    authorization_url = await auth_flow.start(context=authorization_context);

    print("✅ Authorization URL generated!");
    print(f"\n{authorization_url}");
    print("\n" + "="*80);

    # Display clickable button
    html_button = f"""
    <div style="margin: 20px 0;">
        <a href="{authorization_url}" target="_blank" style="
            display: inline-block;
            padding: 15px 30px;
            background-color: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: bold;
            font-size: 16px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.2);
        ">Click Here to Authenticate with Okta</a>
    </div>
    <div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
        <strong>Instructions:</strong>
        <ol style="margin: 10px 0 0 0;">
            <li>Click the button above to open the authorization URL in a new tab</li>
            <li>Sign in with your Okta credentials</li>
            <li>After authentication, you'll be redirected to: <code>{REDIRECT_URI}</code> which will fail</li>
            <li>Copy the entire URL of failed redirect from browser URL bar
            <li>Paste the code in the next cell to exchange it for tokens</li>
        </ol>
    </div>
    """

    display(HTML(html_button));

    print("\nWhat to do next:");
    print("   1. Click the button above");
    print("   2. Sign in to Okta");
    print("   3. Copy the entire URL from failed redirect in browser");
    print("   4. Paste it in the next cell");
    print("\nNote: The ID token will be issued by the Org Authorization Server");
    print(f"      Issuer: {OKTA_DOMAIN}");

  except Exception as e:

    print(f"❌ ERROR: {e}");
    raise

await authorize()

### ② Exchange Authorization Code for Tokens

After authenticating...
1. copy the entire URL
1. paste the URL below
1. and run this cell to obtain tokens

In [ ]:
# @title { display-mode: "form" }

import jwt

REDIRECT_URL = '' # @param { type: "string", placeholder: "Insert entire URL string here."}

async def exchange_code_for_tokens():
    if not REDIRECT_URL:
        print("\n❌ No url provided! Please paste the entire URL containing the `code` and run this cell again.");
        return

    if "auth_flow" not in globals() and not STATE.get("auth_flow"):
            raise SystemExit("❌ Missing auth flow. Run Cell 14 (Build and Open Authorization URL) first.");

    print("\n" + "=" * 80);
    print("② Exchange Authorization Code for User Tokens");
    print("=" * 80);

    try:
        flow = STATE.get("auth_flow", auth_flow);
        token = await flow.resume(REDIRECT_URL);

        print("Token exchange successful!\n");
        print("Token Response:");
        print(f"   Token Type: {token.token_type}");
        print(f"   Expires In: {token.expires_in} seconds");
        print(f"   Scope: {token.scope}");

        # Extract tokens
        global ID_TOKEN, ACCESS_TOKEN, REFRESH_TOKEN
        ID_TOKEN = token.id_token.raw if token.id_token else None
        ACCESS_TOKEN = token.access_token
        REFRESH_TOKEN = token.refresh_token

        STATE["id_token"] = ID_TOKEN;
        STATE["access_token"] = ACCESS_TOKEN;
        STATE["refresh_token"] = REFRESH_TOKEN;

        print(f"\nTokens Obtained:");
        if ID_TOKEN:
            print(f" ✅ ID Token: https://jwt.io#token={ID_TOKEN}");
        if ACCESS_TOKEN:
            print(f" ✅ Access Token: https://jwt.io#token={ACCESS_TOKEN}");
        if REFRESH_TOKEN:
            print(f" ✅ Refresh Token: {REFRESH_TOKEN[:50]}...");

        # Decode and display ID token claims (optional)
        if ID_TOKEN:
            decoded = jwt.decode(ID_TOKEN, options={"verify_signature": False});

            print(f"\nID Token Claims:");
            print(f"   Subject: {decoded.get('sub')}");
            print(f"   Email: {decoded.get('email', 'N/A')}");
            print(f"   Name: {decoded.get('name', 'N/A')}");
            print(f"   Issuer: {decoded.get('iss')}");
            print(f"   Audience: {decoded.get('aud')}");

            # Verify issuer is the Okta domain (Org Authorization Server)
            if decoded.get('iss') == OKTA_DOMAIN:
                print(f"\n ✅ Token issued by Org Authorization Server: {OKTA_DOMAIN}");
            else:
                print(f"\n ⚠️ Unexpected issuer: {decoded.get('iss')}");
                print(f"   Expected: {OKTA_DOMAIN}");

            print("Please verify your configuration!");
            print("=" * 60);
            print(f"   Okta Domain: {OKTA_DOMAIN}");
            print(f"   Principal ID: {PRINCIPAL_ID}");

            print("=" * 60)

    except Exception as e:
        print(f" ❌ Error during token exchange: {e}");
        print("\nTroubleshooting:");
        print("   • Make sure you copied the entire authorization code");
        print("   • Verify your redirect_uri matches what's registered in Okta");
        print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)");
        print("   • Ensure your client_id and client_secret are correct");

await exchange_code_for_tokens()

---
## Step 2: Exchange user token for resource access token

This step can result in **three** possible scenarios:

1. User has a valid refresh token → Okta will obtain a new access token. 
2. No valid token(s) are available → Okta will return in `interaction_uri`.
   - In this scenario, click on the button to authenticate then **rerun** this step.
3. User has a valid access token → Okta will return the access token.

### Prerequisites
- Setup and Step 1 have been run at least once.
- User's token and `okta_sdk` are available in memory.
- `RESOURCE_INDICATOR` points to the resource connected in Okta.

### Output
- Stores the exchanged resource access token in `RESOURCE_TOKEN`.

### Sequence / Flow

[![](https://mermaid.ink/img/pako:eNrFVV1vmzAU_SuW97KqpAqfDTxUQnSa-tBlSpo9TLw45IZYISY1puoa5b_v2oSUpLRan4aQgHvP_Toc2zualQugEa3gsQaRwS1nuWSbVKSK1aoU9WYOknj6e8uk4hnfMqHIjLCKzCqQ545EO5KCA77H2-25O9buONfen5ILbS3OMeN4qlHjtWJkLHMS12r1QqYgn96WmzTYCVRlLTM4xyJaQqaIzOdf7euhRfB2fP9CO1L1o1RAJM9XipRLkkRkqmBLnMaZqmQwuLmJI0yO1FSKXJqByd0teSjXIFpYjChsOSLfnrMVEzmcwsiylK_9HSJJ58LYNsN3wFZbqDoUOQGzQhGWZVBVjZ9w8cQKvjhFtUjjwoxLTLpqE5Ke60gSkmNIsu3woh9qSCtgaTgzXU8zEEzyktj9Ec2AEw2dHFrpjtAfhPBBS0vcoBs6L49JHt6P7nA6VaU8kFn1g0H00AdFBQRXwAqlyjOmeCmQpceaS-hBH_kLkTsvRAo_Yu8ouVmHPOf9SRoZekMbh78TCiTLTD-zyV1_UKxDkk-FGLFjPz8LYKeTQ3_ArCkRd4BYKylFpRf3e0XaP_qZqI5-plzkBeAjF2T8f5XTrxptPDUbIX2gGL3Y_ECvOKdHM6d6iTt6cd-CX7VidqDu8P_QPxrMU-9r-E4tmku-oJGSNVh0A3LD9Cfd6bCU4g_cQEojfF0wuU6p1dj1xz2eKdplQjvwX9g5mxdQaeeuKZ_SOcvWuSxrbMik--K6bkq1d5-KPfaBO_3vsty0rSA0X9FoyZBZi9bbBYrocGwdrRInAJlgUkUj3w9dk4VGO_pMo3B05Vx7_sgLQtcO7QCdf2g0sP2rwPFsL7RDx_WHQRDsLfpiCttXQ9tFYzAaOcPQd91ri8KCo0Dum-PTnKL7vxXNBHM?type=png)](https://mermaid.live/edit#pako:eNrFVV1vokAU_SuT2Zdtig0fosJDE0I3mz50bbTuw4aXEa84EQc7DE23xv_eO4NYtLTZPi0xEeaecz8OZ5gdTYsF0JCW8FiBSOGGs0yyTSISxSpViGozB0n6-nnLpOIp3zKhyIywksxKkOeBWAfinAPeR9vteTjS4SjT0XvJhV7NzzHjaKpR47ViZCwzElVq9UKmIJ_el5vU2AmURSVTOMciWkKqiMzm352hbRH8ub5_oQOJ-lUoIJJnK0WKJYlDMlWwJW4dTFTc611fRyEmR2lKRS7NwOT2hjwUaxANLEIUthySH8_piokMTmFkWci3_g5M0rqQ22T4CdhqA1WHIidglivC0hTKso4TLp5YzhenqAZpQphxiUlXTULScR1FQnGMSI4TXHRDjWg5LI1mputpCoJJXhCnm1EPONHQyaGV9gjdJIT3GlmiGl3LeXlM8vAxu6XpVBXyIGbZDQbRIR_kJRDcASu0Kk-Z4oVAlR4rLqEDfdQvQO36AUr4mXpHy81a4rkfT1LbsG87OPytUCBZavqZTW67SZGmxF-iGLNjP_c5sNPJoZswq0tELSDWigtR6s39UZHmjX6F1fLPlIssB_zLBBn_X-d0u0Yvni4bI33iGL3Z_IHecW6HZ079ErX84r0Hv3nFfIHaw_9D_7hg_vV3De-pRTPJFzRUsgKLbkBumH6kO01LKL7ADSQ0xNsFk-uEWvW6frjDM0WHDLUF_42ds3kOpQ7u6vIJnbN0ncmiwoZMum-e5yVUR_eJ2GMf-KX_UxSbphWEZisaLhkqa9Fqu0ATHY6t46rECUDGmFTR0Pc9z2Sh4Y4-09Cx7St32PeHvhP43sh2fIv-pWHPvRq4fafvBu4oCFwnCAZ7i76Yys6V7Xiu5w9GI9dGkje0KCw4OuSuPj_NMbp_BQ5OBJY)

In [ ]:
from IPython.display import HTML, display
from okta_client.authfoundation import OAuth2Error, RefreshTokenFlow
from okta_client.oauth2auth.token_exchange import TokenExchangeFlow, TokenType

RESOURCE_TOKEN = None

if not STATE.get("id_token") and "ID_TOKEN" not in globals():
    raise SystemExit("❌ Missing ID_TOKEN. Run Step 1 first.");

if "okta_sdk" not in globals() and not STATE.get("okta_sdk"):
    raise SystemExit("❌ Missing Okta SDK. Run Setup Step 3 (Initialize the Okta SDK) first.")

async def refresh_tokens_if_available():
    print("\n" + "=" * 80)
    print("Ensure user token is fresh")
    print("=" * 80)

    refresh_token = STATE.get("refresh_token", globals().get("REFRESH_TOKEN"));

    if not refresh_token:
        print("⚠️ No REFRESH_TOKEN available. Continuing with current ID_TOKEN.")
        print("If this token is expired, rerun Step 1 to get a fresh ID token.")

        return STATE.get("id_token", globals().get("ID_TOKEN"));

    try:
        print("⑦ Refreshing tokens using REFRESH_TOKEN...");
        refreshed_result = await RefreshTokenFlow(client=okta_sdk).start(refresh_token)

        global ID_TOKEN, REFRESH_TOKEN
        if refreshed_result.id_token:
            ID_TOKEN = refreshed_result.id_token.raw
            STATE["id_token"] = ID_TOKEN
        if refreshed_result.refresh_token:
            REFRESH_TOKEN = refreshed_result.refresh_token
            STATE["refresh_token"] = REFRESH_TOKEN

        print("✅ Tokens refreshed successfully")
        print(f"   Expires In: {refreshed_result.expires_in} seconds")
        print(f"   Scope: {refreshed_result.scope or 'N/A'}")
        return STATE.get("id_token") or globals().get("ID_TOKEN")
    except Exception as e:
        print(f"⚠️ [WARNING] Refresh failed: {e}")
        print("Continuing with existing ID_TOKEN. If exchange fails, rerun Step 1.")
        return STATE.get("id_token") or globals().get("ID_TOKEN")

async def exchange_for_resource_token(id_token: str):

    if not id_token:
        raise SystemExit("❌ Missing ID token for token exchange.")

    try:
        flow = TokenExchangeFlow(client=okta_sdk)

        print("\n⑤ Exchanging ID token for resource access token...")

        token_result = await flow.start(
            subject_token=id_token,
            subject_token_type=TokenType.ID_TOKEN,
            resource=[RESOURCE_INDICATOR],
            requested_token_type="urn:okta:params:oauth:token-type:oauth-sts",
        )

        print("✅ Resource access token exchange successful")
        print(f"   Token Type: {token_result.token_type}")
        print(f"   Expires In: {token_result.expires_in} seconds")
        print(f"   Scope: {token_result.scope or 'N/A'}")

        return token_result.access_token

    except OAuth2Error as e:
        if e.error == "interaction_required":
            print("⚠️ ①⓪ Interaction required for token exchange.")

            interaction_uri = e.additional_fields.get("interaction_uri", None) or globals().get("INTERACTION_URI")

            if interaction_uri:
                print("⚠️ Additional user interaction is required. Authorize, then rerun Step 2.")

                html_button = f"""
                <div style="margin: 20px 0;">
                    <a href="{interaction_uri}" target="_blank" style="
                        display: inline-block;
                        padding: 15px 30px;
                        background-color: #007bff;
                        color: white;
                        text-decoration: none;
                        border-radius: 5px;
                        font-weight: bold;
                        font-size: 16px;
                        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                    ">Click Here to Authorize your Resource</a>
                </div>
                <div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
                    <strong>Instructions:</strong>
                    <ol style="margin: 10px 0 0 0;">
                        <li>Click the button above to open authorization in a new tab.</li>
                        <li>Complete sign-in/consent.</li>
                        <li>Return here and rerun Step 2.</li>
                    </ol>
                </div>
                """
                display(HTML(html_button))
                return None

    except Exception as e:
        raise SystemExit("❌ Token exchange failed. Please check your configuration and try again.")

print("\n" + "=" * 80)
print("Step 2: Exchange ID token for resource access token")
print("=" * 80)

current_id_token = await refresh_tokens_if_available()

if current_id_token:
    RESOURCE_TOKEN = await exchange_for_resource_token(current_id_token);
    STATE["resource_token"] = RESOURCE_TOKEN;

    if RESOURCE_TOKEN:
        print("🎉  RESOURCE_TOKEN is now set for Step 3")
        print(f"   Token preview (if token is a JWT): https://jwt.io#token={RESOURCE_TOKEN}")

---
## Step 3: Use Resource Access Token to Call Resource API

This step uses `RESOURCE_TOKEN` from Step 2 to call your resource's endpoint.

_You will need to configure the endpoint in order for this step to work._

### Hard-fail behavior
- If `RESOURCE_TOKEN` is missing, this step fails with remediation instructions.
- If the resource returns `401`, rerun Step 3 to obtain a fresh final resource access token.
- If the resource returns `403`, verify the connected app granted the required email scope.

[![](https://mermaid.ink/img/pako:eNp9Us1u2zAMfhWCu7SoE1hWnNg6FDDaa5chGXYYfFFs1jESWxlNF1uDvPukeN2KHqaTyO-HlMgzVq4mNDjQj5H6ih5b27Dtyr4UO4rrx25HDCoPiZNlaav2ZHuBAuwARUP--oXbPmSPHznrYhtY64NYWHMDxSj7V9gSvxB_5G4m7oYGN3JF_-e-Z3rQw0yVADe7G51FsNIRqDi_DUApn50QcNvsBdwzFAa2QifQE1jM7u83xpv51w8C4v51cAdFVdEwwFd3IF8E_pzNzGuKoBnGowQb6muMsOG2RiM8UoQdcWdDiOcgLFH21FGJxl9ry4cSoykfgic_gQBdpe_o3yy3dnekIYDnqYESd7Y6NOzGvp7sPmmtSwzopewvvg__Rd-d695a8dRmj-bZHgcfjafaytuQ_2bZv4D4wZsKmjRVq6sLmjP-RKPieJ6sFukqVXmqs1ilEf5CM0vmy2ShFkmeZHmeqDxfXiJ8vVZW81jpRKfLLEtiL9LekOpWHD9N23Zdustv5U7PMw?type=png)](https://mermaid.live/edit#pako:eNp9Us9vmzAU_lee3i6tSiKMQwI-VELttcuUTDtMXBx4JSgBZ49HtTXK_z47rFvVwzj5fb_8GfuMlasJDQ70Y6S-osfWNmy7si_FjuL6sdsRg8o94KGTZWmr9mR7gQLsAEVDfvmF2z6gx4-adbENqvVBLKy5gWKU_StsiV-IP2o3k3ZDgxu5ov9r3yunakyVADe7G51FsNIRqDi_DUQpn50QcNvsBdwzFAa2QifQE1nM7u83xof58w8C4v41uIOiqmgY4Ks7kN8E_nybmfcUwTOMRwkx1NcYYcNtjUZ4pAg74s6GEc_BWKLsqaMSjV_Wlg8lRhMehid_B4G6Wt_Jv1lu7e5IQyDPU4ESd7Y6NOzGvp7iPmmtSwzspewvvof_Rd-d696qeGmzR_Nsj4OfxlNt5e2a_6LsT0D84EMFTZoqfU1Bc8afaFQcz5PVIl2lKk91Fqs0wl9oZsl8mSzUIsmTLM8TlefLS4Sv153VPFY60ekyy5LYm_QqQqpbcfw0vbfrs7v8BtmUz_k)

In [ ]:
import http.client
import json

# @title { display-mode: "form" }
# @markdown <hr>
# @markdown <br><em>You may need to modify this code in order to call your specific resource.</em>
# @markdown <br><br><em>We have provided some sample code for each resource type outlined in the sample guide. Adjust the code accordingly.</em><br><br>
# @markdown <br>
# @markdown <hr>

global RESOURCE_PROVIDER
global RESOURCE_URL
global RESOURCE_ENDPOINT
global RESOURCE_BASE_URL

# @markdown Select provider mode. Use <b>auto</b> to infer from filled fields.
RESOURCE_PROVIDER = 'auto' # @param ["auto", "custom", "atlassian", "snow", "github", "msft", "slack"]
RESOURCE_URL = '' # @param {"type":"string","placeholder":"Enter the resource base URL"}
RESOURCE_ENDPOINT = '' # @param {"type":"string","placeholder":"Enter the resource path URL."}

# @markdown <hr>
# @markdown Atlassian Jira
ATLASSIAN_SITE_URL = '' # @param {"type":"string","placeholder":"Enter your Atlassian site URL (e.g., yoursite.atlassian.net)"}
ATLASSIAN_ENDPOINT = '/rest/api/3/project' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown ServiceNow
SNOW_INSTANCE = '' # @param {"type":"string","placeholder":"Enter your ServiceNow instance name (e.g., dev12345)"}
SNOW_URL = f'{SNOW_INSTANCE}.service-now.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
SNOW_ENDPOINT = '/api/now/table/incident' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Github
GITHUB_URL = 'api.github.com' # @param {"type":"string","placeholder":"Enter your Github URL (if different)"}
GITHUB_ENDPOINT = '/user/emails' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Office 365
MSFT_URL = 'graph.microsoft.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
MSFT_ENDPOINT = '/v1.0/me' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Slack
SLACK_URL = 'slack.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
SLACK_ENDPOINT = '/api/conversations.list' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

resource_token = STATE.get("resource_token", globals().get("RESOURCE_TOKEN"));

if not resource_token:
    raise SystemExit(
        "❌ RESOURCE_TOKEN is missing. Rerun Step 2 to complete the resource access token exchange."
    )

def _normalize_url(url: str) -> str:
    return (url or '').strip().replace('https://', '').replace('http://', '').rstrip('/')

def _normalize_endpoint(endpoint: str) -> str:
    endpoint = (endpoint or '').strip()
    if endpoint and not endpoint.startswith('/'):
        endpoint = f"/{endpoint}"
    return endpoint

provider = (RESOURCE_PROVIDER or 'auto').strip().lower()
allowed = {'auto', 'custom', 'atlassian', 'snow', 'github', 'msft', 'slack'}
if provider not in allowed:
    raise ValueError(f"Invalid RESOURCE_PROVIDER '{RESOURCE_PROVIDER}'. Must be one of: {sorted(allowed)}")

resolved_from = None
resolved_url = ''
resolved_endpoint = ''

if provider == 'custom':
    resolved_url = RESOURCE_URL
    resolved_endpoint = RESOURCE_ENDPOINT
    resolved_from = 'custom provider selection'
elif provider == 'atlassian':
    resolved_url = ATLASSIAN_SITE_URL
    resolved_endpoint = RESOURCE_ENDPOINT or ATLASSIAN_ENDPOINT
    resolved_from = 'Atlassian provider selection'
elif provider == 'snow':
    resolved_url = SNOW_URL
    resolved_endpoint = RESOURCE_ENDPOINT or SNOW_ENDPOINT
    resolved_from = 'ServiceNow provider selection'
elif provider == 'github':
    resolved_url = GITHUB_URL
    resolved_endpoint = RESOURCE_ENDPOINT or GITHUB_ENDPOINT
    resolved_from = 'Github provider selection'
elif provider == 'msft':
    resolved_url = MSFT_URL
    resolved_endpoint = RESOURCE_ENDPOINT or MSFT_ENDPOINT
    resolved_from = 'MSFT provider selection'
elif provider == 'slack':
    resolved_url = SLACK_URL
    resolved_endpoint = RESOURCE_ENDPOINT or SLACK_ENDPOINT
    resolved_from = 'Slack provider selection'
else:
    # Auto priority: explicit custom -> Atlassian -> ServiceNow -> Github
    if RESOURCE_URL and RESOURCE_ENDPOINT:
        resolved_url = RESOURCE_URL
        resolved_endpoint = RESOURCE_ENDPOINT
        resolved_from = 'auto: explicit RESOURCE_URL/RESOURCE_ENDPOINT'
    elif ATLASSIAN_SITE_URL:
        resolved_url = ATLASSIAN_SITE_URL
        resolved_endpoint = RESOURCE_ENDPOINT or ATLASSIAN_ENDPOINT
        resolved_from = 'auto: Atlassian settings'
    elif SNOW_INSTANCE:
        resolved_url = SNOW_URL
        resolved_endpoint = RESOURCE_ENDPOINT or SNOW_ENDPOINT
        resolved_from = 'auto: ServiceNow settings'
    else:
        resolved_url = GITHUB_URL
        resolved_endpoint = RESOURCE_ENDPOINT or GITHUB_ENDPOINT
        resolved_from = 'auto: Github defaults'

RESOURCE_URL = _normalize_url(resolved_url)
RESOURCE_ENDPOINT = _normalize_endpoint(resolved_endpoint)

if not RESOURCE_URL or not RESOURCE_ENDPOINT:
    raise ValueError(
        "❌ Unable to resolve resource target. Set RESOURCE_PROVIDER to a specific provider, "
        "or provide RESOURCE_URL and RESOURCE_ENDPOINT for custom mode."
    )

RESOURCE_BASE_URL = RESOURCE_URL  # Backward-compatible alias

print("\n" + "=" * 80)
print(f"STEP 3: Use resource access token to call {RESOURCE_ENDPOINT}")
print("=" * 80)
print(f"Resolved target from: {resolved_from}")
print(f"Resolved base URL: {RESOURCE_URL}")
print(f"①⑨ Making API call to resource endpoint {RESOURCE_ENDPOINT}...")

conn = http.client.HTTPSConnection(RESOURCE_URL)
headers = {
    "Accept": "application/json",
    "User-Agent": "okta-ai-poc/1.0",
    "Authorization": f"Bearer {resource_token}"
}

if 'api.github.com' in RESOURCE_URL:
    headers['X-GitHub-Api-Version'] = '2022-11-28'

conn.request("GET", RESOURCE_ENDPOINT, "", headers)
res = conn.getresponse()
raw_data = res.read().decode("utf-8")

print(f"Response Status: {res.status} {res.reason}")

if res.status >= 400:
    detail = raw_data
    try:
        detail = json.dumps(json.loads(raw_data), indent=2)
    except Exception:
        pass

    remediation = "Verify the connected app granted the required permissions/scopes for this API."
    if res.status == 401:
        remediation = "Resource rejected the token. Rerun Step 2 to obtain a fresh resource access token."

    raise RuntimeError(
        "Resource API call failed. "
        f"{remediation} "
        f"Status={res.status}. Response={detail}"
    )

try:
    parsed = json.loads(raw_data)
except Exception:
    parsed = {"raw": raw_data}

print("Response from resource API:")
print(json.dumps(parsed, indent=2))

---

## Resources

- [Okta AI SDK Documentation](https://github.com/okta/okta-client-python/tree/main)

---
